# FoodLence OCR Noise Augmentation

This notebook creates deterministic OCR-noise variants while retaining the parent sample, reviewed annotation label, and source identifiers.

**Data contract:** `ml/data/raw/ocr_samples.csv` contains 20 raw captures with image_path and raw_ocr_text columns; it has no ingredient labels or stable product source IDs. Labels are joined from `ml/data/annotated/ocr_ingredients.csv`, using only records marked `reviewed`. The cleanest available ingredient phrase is the reviewed annotation's `text`; the original OCR capture is retained separately as `raw_ocr_text`. Generated variants inherit a reviewed label but are themselves synthetic and are not independently human-reviewed.

Before generation, place a disjoint, labeled evaluation file at `ml/data/test/ocr_ingredients.csv`. This notebook excludes training annotations from any capture IDs in that holdout and rejects exact normalized phrase overlap. It refuses to generate data when source rows, reviewed annotations, or the holdout are missing.

## Cell 1 — Imports and reproducible setup

In [ ]:
import csv
import hashlib
import json
import math
import random
import re
import unicodedata
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "ml" / "data").is_dir():
    raise RuntimeError("Run this notebook from the FoodLence repository root.")

RAW_PATH = ROOT / "ml/data/raw/ocr_samples.csv"
ANNOTATED_PATH = ROOT / "ml/data/annotated/ocr_ingredients.csv"
TEST_PATH = ROOT / "ml/data/test/ocr_ingredients.csv"
OUTPUT_PATH = ROOT / "ml/data/augmented/variants.csv"

RANDOM_SEED = 5104
MIN_VARIANTS = 3000
MAX_VARIANTS_PER_ANNOTATION = 200
rng = random.Random(RANDOM_SEED)

print(f"Repository: {ROOT}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Variant target: {MIN_VARIANTS:,}")

## Cell 2 — Load OCR captures, reviewed labels, and holdout

In [ ]:
def read_csv_rows(path: Path, required_columns: set[str]) -> list[dict[str, str]]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required input is missing: {path}. Do not generate variants without it."
        )
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        columns = set(reader.fieldnames or [])
        missing = sorted(required_columns - columns)
        if missing:
            raise ValueError(f"{path.name} is missing required columns: {missing}")
        rows = list(reader)
    if not rows:
        raise ValueError(
            f"{path} contains a header but no records. Add real source/review data first; "
            "the notebook will not fabricate examples."
        )
    return rows


raw_source_rows = read_csv_rows(
    RAW_PATH, {"image_path", "raw_ocr_text"}
)
raw_rows = []
for row in raw_source_rows:
    capture_id = row["image_path"].strip()
    if not capture_id:
        raise ValueError("Every raw capture needs a non-empty image_path.")
    raw_rows.append({
        **row, "sample_id": capture_id, "source_id": capture_id,
        "ocr_text": row["raw_ocr_text"],
    })
annotation_rows = read_csv_rows(
    ANNOTATED_PATH,
    {"sample_id", "source_id", "text", "label", "entity_type",
     "is_precautionary", "annotation_status", "annotator"},
)
test_rows = read_csv_rows(
    TEST_PATH, {"sample_id", "text", "label", "annotation_status"}
)
if any(row["annotation_status"].strip().casefold() != "reviewed" for row in test_rows):
    raise ValueError("Every holdout label must have reviewed status.")

print(f"Raw OCR captures: {len(raw_rows):,}")
print(f"Annotation rows: {len(annotation_rows):,}")
print(f"Independent holdout rows: {len(test_rows):,}")

## Cell 3 — Link reviewed ingredient annotations to captures and exclude holdout groups

In [ ]:
def truthy(value: str) -> bool:
    return str(value or "").strip().casefold() in {"1", "true", "yes", "y"}


def normalize_for_leakage(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text or "")).casefold()
    return re.sub(r"\s+", " ", text).strip()


raw_by_id: dict[str, dict[str, str]] = {}
for row in raw_rows:
    sample_id = row["sample_id"].strip()
    if not sample_id:
        raise ValueError("Every raw OCR row must have a non-empty sample_id.")
    if sample_id in raw_by_id:
        raise ValueError(f"Duplicate raw sample_id: {sample_id}")
    raw_by_id[sample_id] = row

test_sample_ids = {row["sample_id"].strip() for row in test_rows}
test_source_ids = {row["source_id"].strip() for row in test_rows if row.get("source_id", "").strip()}
if not test_sample_ids:
    raise ValueError("The holdout must contain sample_id values.")
missing_test_samples = sorted(test_sample_ids - set(raw_by_id))
if missing_test_samples:
    raise ValueError(
        f"Holdout sample IDs are absent from raw OCR captures: {missing_test_samples[:10]}"
    )

test_texts = {
    normalize_for_leakage(row["text"])
    for row in test_rows
    if normalize_for_leakage(row["text"])
}
training_examples = []
excluded_holdout_rows = 0
unreviewed_rows = 0

for row in annotation_rows:
    if row["annotation_status"].strip().casefold() != "reviewed":
        unreviewed_rows += 1
        continue
    sample_id = row["sample_id"].strip()
    if sample_id not in raw_by_id:
        raise ValueError(f"Reviewed annotation references unknown sample_id: {sample_id}")
    capture = raw_by_id[sample_id]
    if row.get("source_id", "").strip() and row["source_id"].strip() != capture["source_id"].strip():
        raise ValueError(f"source_id mismatch for sample_id {sample_id}")
    if sample_id in test_sample_ids or row["source_id"].strip() in test_source_ids:
        excluded_holdout_rows += 1
        continue
    if truthy(row["is_precautionary"]):
        continue
    if row["entity_type"].strip().casefold() not in {"ingredient", "additive"}:
        continue
    text = row["text"].strip()
    label = row["label"].strip()
    annotator = row["annotator"].strip()
    if not text or not label or not annotator:
        raise ValueError(f"Reviewed annotation is missing text, label, or annotator: {sample_id}")
    training_examples.append({
        "original_id": sample_id,
        "source_id": row["source_id"].strip(),
        "original_text": text,
        "raw_ocr_text": capture["ocr_text"],
        "label": label,
        "entity_type": row["entity_type"].strip(),
        "is_precautionary": "false",
        "parent_annotation_status": "reviewed",
        "parent_annotator": annotator,
    })

if not training_examples:
    raise RuntimeError("No reviewed, non-precautionary ingredient/additive annotations remain for training.")

distinct_parents = {
    (row["original_id"], row["original_text"], row["label"])
    for row in training_examples
}
minimum_parents = math.ceil(MIN_VARIANTS / MAX_VARIANTS_PER_ANNOTATION)
if len(distinct_parents) < minimum_parents:
    raise RuntimeError(
        f"Only {len(distinct_parents)} reviewed training annotations remain; at least "
        f"{minimum_parents} are required to produce {MIN_VARIANTS:,} variants while "
        f"capping each annotation at {MAX_VARIANTS_PER_ANNOTATION} variants."
    )

training_source_ids = {row["source_id"] for row in training_examples}
if training_source_ids & test_source_ids:
    raise AssertionError("Training and evaluation source_id groups overlap.")

print(f"Reviewed training annotations: {len(training_examples):,}")
print(f"Excluded because they belong to the holdout: {excluded_holdout_rows:,}")
print(f"Skipped unreviewed/ambiguous/rejected rows: {unreviewed_rows:,}")
print(f"Distinct training source groups: {len(training_source_ids):,}")

## Cell 4 — Controlled OCR-noise operations

In [ ]:
OCR_ERRORS = {
    "o": ["0", "q"],
    "i": ["1", "l"],
    "l": ["1", "i"],
    "s": ["5"],
    "b": ["8", "6"],
    "a": ["4", "@"],
    "e": ["3"],
    "g": ["9", "6"],
    "t": ["7"],
    "p": ["9", "q"],
    "q": ["9", "p"],
    "c": ["("],
}
DIGIT_CONFUSABLES = {
    "0": ["O"], "1": ["I", "l"], "2": ["Z"], "5": ["S"],
    "6": ["G"], "8": ["B"], "9": ["g"],
}
UNICODE_CONFUSABLES = {
    "a": "а", "e": "е", "o": "о", "p": "р",
    "c": "с", "x": "х", "y": "у", "i": "і",
}
NOISE_OPERATIONS = [
    "char_substitution", "space_insertion", "space_removal",
    "digit_swap", "mixed_case", "unicode_confusables",
]


def apply_char_substitution(text: str, randomizer: random.Random) -> str:
    result = []
    for char in text:
        choices = OCR_ERRORS.get(char.casefold(), [])
        if choices and randomizer.random() < 0.22:
            replacement = randomizer.choice(choices)
            result.append(replacement.upper() if char.isupper() else replacement)
        else:
            result.append(char)
    return "".join(result)


def add_random_spaces(text: str, randomizer: random.Random, density: float = 0.12) -> str:
    result = []
    for index, char in enumerate(text):
        result.append(char)
        if (
            char.isalnum()
            and index + 1 < len(text)
            and text[index + 1].isalnum()
            and randomizer.random() < density
        ):
            result.append(" ")
    return "".join(result)


def remove_random_spaces(text: str, randomizer: random.Random) -> str:
    positions = [i for i, char in enumerate(text) if char.isspace()]
    if not positions:
        return text
    position = randomizer.choice(positions)
    return text[:position] + text[position + 1:]


def apply_digit_swap(text: str, randomizer: random.Random) -> str:
    result = []
    for char in text:
        choices = DIGIT_CONFUSABLES.get(char)
        if choices and randomizer.random() < 0.35:
            result.append(randomizer.choice(choices))
        else:
            result.append(char)
    return "".join(result)


def flip_random_case(text: str, randomizer: random.Random) -> str:
    return "".join(
        char.swapcase() if char.isalpha() and randomizer.random() < 0.35 else char
        for char in text
    )


def apply_unicode_confusables(text: str, randomizer: random.Random) -> str:
    return "".join(
        UNICODE_CONFUSABLES.get(char.casefold(), char)
        if char.isalpha() and randomizer.random() < 0.10
        else char
        for char in text
    )


def generate_variant(text: str, operations: list[str], randomizer: random.Random) -> str:
    for operation in operations:
        if operation == "char_substitution":
            text = apply_char_substitution(text, randomizer)
        elif operation == "space_insertion":
            text = add_random_spaces(text, randomizer)
        elif operation == "space_removal":
            text = remove_random_spaces(text, randomizer)
        elif operation == "digit_swap":
            text = apply_digit_swap(text, randomizer)
        elif operation == "mixed_case":
            text = flip_random_case(text, randomizer)
        elif operation == "unicode_confusables":
            text = apply_unicode_confusables(text, randomizer)
        else:
            raise ValueError(f"Unknown noise operation: {operation}")
    return text

## Cell 5 — Generate deterministic variants while preserving parent labels

In [ ]:
def stable_variant_id(original_id: str, source_id: str, label: str, variant_text: str, index: int) -> str:
    payload = "\\0".join([original_id, source_id, label, variant_text, str(index)])
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:24]


def generate_augmented_variants(
    examples: list[dict[str, str]],
    min_total: int = MIN_VARIANTS,
    random_seed: int = RANDOM_SEED,
) -> list[dict[str, str]]:
    if not examples:
        raise ValueError("No training examples supplied.")
    if len(examples) < math.ceil(min_total / MAX_VARIANTS_PER_ANNOTATION):
        raise ValueError("Too few distinct reviewed annotations for the configured per-annotation cap.")

    randomizer = random.Random(random_seed)
    quotient, remainder = divmod(min_total, len(examples))
    variants = []
    for parent_index, original in enumerate(examples):
        count = quotient + (1 if parent_index < remainder else 0)
        seen_for_parent = {normalize_for_leakage(original["original_text"])}
        attempts = 0
        made = 0
        max_attempts = max(1000, count * 100)
        while made < count and attempts < max_attempts:
            attempts += 1
            operation_count = randomizer.randint(1, min(4, len(NOISE_OPERATIONS)))
            operations = randomizer.sample(NOISE_OPERATIONS, k=operation_count)
            variant_text = generate_variant(original["original_text"], operations, randomizer)
            normalized = normalize_for_leakage(variant_text)
            if not normalized or normalized in seen_for_parent:
                continue
            seen_for_parent.add(normalized)
            variants.append({
                "variant_id": stable_variant_id(
                    original["original_id"], original["source_id"], original["label"],
                    variant_text, made,
                ),
                "original_id": original["original_id"],
                "source_id": original["source_id"],
                "variant_text": variant_text,
                "original_text": original["original_text"],
                "raw_ocr_text": original["raw_ocr_text"],
                "label": original["label"],
                "entity_type": original["entity_type"],
                "is_precautionary": original["is_precautionary"],
                "variant_source": "augmented",
                "variation_operations": "|".join(operations),
                "review_status": "synthetic_variant_inherits_reviewed_parent_label",
                "parent_annotation_status": original["parent_annotation_status"],
                "parent_annotator": original["parent_annotator"],
                "normalized_original": normalize_for_leakage(original["original_text"]),
                "normalized_variant": normalized,
            })
            made += 1
        if made != count:
            raise RuntimeError(
                f"Generated {made}/{count} unique variants for parent "
                f"{original['original_id']}; refusing to silently under-deliver."
            )
    return variants

## Cell 6 — Validate leakage and write the augmented CSV

In [ ]:
variants = generate_augmented_variants(training_examples)

variant_ids = [row["variant_id"] for row in variants]
if len(variant_ids) != len(set(variant_ids)):
    raise AssertionError("variant_id values are not unique.")
if len(variants) < MIN_VARIANTS:
    raise AssertionError(f"Expected at least {MIN_VARIANTS:,} variants, got {len(variants):,}.")

variant_texts = {row["normalized_variant"] for row in variants}
phrase_overlap = variant_texts & test_texts
variant_source_ids = {row["source_id"] for row in variants}
source_overlap = variant_source_ids & test_source_ids
parent_overlap = {row["original_id"] for row in variants} & test_sample_ids
if phrase_overlap or source_overlap or parent_overlap:
    raise RuntimeError(
        "Leakage audit failed; no output written. "
        f"phrase_overlap={len(phrase_overlap)}, "
        f"source_overlap={len(source_overlap)}, parent_overlap={len(parent_overlap)}"
    )
if any(not row["label"] for row in variants):
    raise AssertionError("At least one generated variant has an empty label.")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fieldnames = [
    "variant_id", "original_id", "source_id", "variant_text", "original_text",
    "raw_ocr_text", "label", "entity_type", "is_precautionary", "variant_source",
    "variation_operations", "review_status", "parent_annotation_status",
    "parent_annotator", "normalized_original", "normalized_variant",
]
with OUTPUT_PATH.open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(variants)

print(f"Wrote {len(variants):,} variants to {OUTPUT_PATH}")
print(f"Unique variant IDs: {len(set(variant_ids)):,}")
print(f"Normalized phrase overlap with holdout: {len(phrase_overlap)}")
print(f"Source-group overlap with holdout: {len(source_overlap)}")

## Cell 7 — Summarize and verify the written dataset

In [ ]:
with OUTPUT_PATH.open("r", encoding="utf-8", newline="") as stream:
    written_rows = list(csv.DictReader(stream))

assert len(written_rows) == len(variants) >= MIN_VARIANTS
assert len({row["variant_id"] for row in written_rows}) == len(written_rows)
assert {row["variant_source"] for row in written_rows} == {"augmented"}
assert all(row["parent_annotation_status"] == "reviewed" for row in written_rows)
assert not ({row["normalized_variant"] for row in written_rows} & test_texts)
assert not ({row["source_id"] for row in written_rows} & test_source_ids)

per_label = Counter(row["label"] for row in written_rows)
print("Augmentation verification passed.")
print(f"Reviewed source annotations used: {len(training_examples):,}")
print(f"Variants generated: {len(written_rows):,}")
print(f"Variants per reviewed label: {dict(sorted(per_label.items()))}")
print(f"Evaluation holdout phrases checked: {len(test_texts):,}")
print(f"Seed: {RANDOM_SEED}")